In [ ]:
# # Given a file path to a CIF file, plots the peaks of that CIF and displays the plot. Peaks are shown as points with vertical dotted lines down to the x-axis.
# def plot_peaks(ciffile: str):
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         structure = Structure.from_file(ciffile)
#         print("Displaying peaks from \"" + os.path.basename(ciffile) + "\":")
#         xrd = XRDCalculator(wavelength = 0.1812, symprec = 0.1)
#         pat = xrd.get_pattern(structure, two_theta_range = (1, 15))
#         # plt.figure(figsize = (10, 6))
#         # plt.plot(pat.x, pat.y, '.')
#         # plt.vlines(pat.x, 0, pat.y, colors='r', linestyles='dashed', alpha = 0.5)
#         # plt.xlabel("2θ (degrees)")  
#         # plt.ylabel("Intensity (a.u.)")
#         # plt.title("Simulated XRD Pattern")  
#         # plt.show()

In [ ]:
# # Display the peaks of both simulated CIF files.
# plot_peaks(ceo2_cif)
# plot_peaks(lab6_cif)

In [ ]:
# # Given a file path to a CIF file, plots a simulated PXRD pattern and displays the plot. Appears as a continuous function.
# # Additionally, changes the size of the lattice to simulate temperature shifts if float sizes are provided.
# # Also, saves the plot data for later use.
# def plot_sim_pattern(ciffile: str, lattice_sizes: tuple[float, float, float] = (0, 0, 0)):
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         print("Displaying simulated pattern for \"" + os.path.basename(ciffile) + "\":")
#         cfg = SimConfig(
#             wavelength = 0.1812,
#             two_theta_range = (0.5, 15.0),
#             n_points = 4096,
#         )
#         size_nm = 380
#         microstrain = 0.001
#         pattern = 0
#         if(lattice_sizes == (0, 0, 0)):
#             pattern = simulate_pattern_from_cif(ciffile, cfg, size_nm = size_nm, microstrain = microstrain)
#         else:
#             pattern = simulate_pattern_from_cif(ciffile, cfg, size_nm = size_nm, microstrain = microstrain, lattice_dims = lattice_sizes)
#         exp_data = np.column_stack((pattern[0], pattern[1]))
#         np.savetxt(os.path.join(fullgraph, os.path.splitext(os.path.basename(ciffile))[0] + "_sim.chi"), exp_data, delimiter = ' ')
#         # plt.figure(figsize = (10, 6))
#         # plt.plot(pattern[0], pattern[1])    
#         # plt.show()

In [ ]:
# # Given a file path to a XY file, plots the given experimental pattern and displays the plot. Appears as a continuous function.
# # Also, saves the plot data for later use.
# def plot_exp_pattern(datfile: str):
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         print("Displaying experimental pattern for \"" + os.path.splitext(os.path.basename(datfile))[0] + "\":")
#         x, y = np.loadtxt(datfile, unpack=True)
#         exp_data = np.column_stack((x, y))
#         np.savetxt(os.path.join(fullgraph, os.path.splitext(os.path.basename(datfile))[0] + "_exp.chi"), exp_data, delimiter = ' ')
#         # plt.figure(figsize=(10, 6))
#         # plt.plot(x, y)
#         # plt.xlabel("2θ (degrees)")  
#         # plt.ylabel("Intensity (a.u.)")
#         # plt.title("Experimental XRD Pattern")  
#         # plt.show()

In [ ]:
# # Code shamelessly stolen and reworked from https://github.com/NSLS2/xpd-profile-collection-ldrd20-31/blob/main/scripts/Matt_multi_phase/Pearson_Notebook%20(1).ipynb.

# # The `simulated_files` list is the list containing graphs of the simulated PXRD patterns.
# simulated_files = [f for f in os.listdir(fullgraph) if f.endswith("sim.chi")]

# # Split off the simulated PXRD patterns to keep the list solely of experimental data.
# experimental_files = [f for f in os.listdir(fullgraph) if f.endswith("exp.chi")]

# # Map the simulated file paths to their data in a dictionary.
# sim_data = {}
# for file_name in simulated_files:
#     path = os.path.join(fullgraph, file_name)
#     x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#     sim_data[file_name] = (x_data, y_data)

# # Prepare a container for the results of the correlation calculations.
# results = []

# # Loop over each file in the experimental files list, and for each file:
# for file in experimental_files:
#     # Load in the experimental file, unpack the data, and split it into an array.
#     path = os.path.join(fullgraph, file)
#     x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#     # Compute the experimental data's correlation to each simulated data.
#     row = {"exp_file": file}
#     for sim_fname, (sim_x, sim_y) in sim_data.items():
#         # Interpolate the simulated data onto the experimental grid.
#         y_interp = np.interp(x_data, sim_x, sim_y)
#         # Calculate the Pearson correlation from the experimental data and interpolated simulated data.
#         correlation = np.corrcoef(y_data, y_interp)[0,1]
#         row[sim_fname] = correlation
#     results.append(row)

# # Print the mixture data for fact-checking.
# print("CIF files in mixture, with respective weights:")
# cif_files = mixture_results.get("chosen_cifs")
# cif_weights = mixture_results.get("weights")
# for i in range(len(cif_files)):
#     print("- " + os.path.basename(cif_files[i]) + ": " + str(cif_weights[i]))

# # Build a dataframe and save.
# df = pd.DataFrame(results).set_index("exp_file")
# df = df.div(df.sum(axis = 1), axis = 0)
# df = df.clip(0, 1)
# print(df.to_string(float_format="%.4f"))

# out_csv = os.path.join(fulldata, "correlation_basic.csv")
# df.to_csv(out_csv)
# print(f"\nSaved correlations to {out_csv}.")

In [ ]:
# # Let's try using Multiple Linear Regression (statistics, not ML) instead of Pearson correlation.

# # The `simulated_files` list is the list containing graphs of the simulated PXRD patterns.
# simulated_files = [f for f in os.listdir(fullgraph) if f.endswith("sim.chi")]

# # Split off the simulated PXRD patterns to keep the list solely of experimental data.
# experimental_files = [f for f in os.listdir(fullgraph) if f.endswith("exp.chi")]

# # Map the simulated file paths to their data in a dictionary.
# sim_data = {}
# for file_name in simulated_files:
#     path = os.path.join(fullgraph, file_name)
#     x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#     sim_data[file_name] = (x_data, y_data)

# # Prepare a container for the results of the correlation calculations.
# results = []

# # Print the mixture data for fact-checking.
# print("CIF files in mixture, with respective weights:")
# cif_files = mixture_results.get("chosen_cifs")
# cif_weights = mixture_results.get("weights")
# for i in range(len(cif_files)):
#     print("- " + os.path.basename(cif_files[i]) + ": " + str(cif_weights[i]))

# # Loop over each file in the experimental files list, and for each file:
# for file in experimental_files:
#     # Load in the experimental file, unpack the data, and split it into an array.
#     path = os.path.join(fullgraph, file)
#     x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#     # Create a dataframe containing all of the available simulated files' data.
#     aggregate_sim_data = pd.DataFrame()
#     for sim_fname, (sim_x, sim_y) in sim_data.items():
#         # Interpolate the simulated data onto the experimental grid.
#         y_interp = np.interp(x_data, sim_x, sim_y)
#         # Add the simulated data onto the aggregate dataframe.
#         aggregate_sim_data.insert(loc = 0, column = sim_fname, value = y_interp)
#     # Set up and fit the stats to an Ordinary Least Squares Model (for statistical MLR).
#     aggregate_sim_data = sm.add_constant(aggregate_sim_data)
#     model_results = sm.OLS(y_data, aggregate_sim_data).fit()
#     # See the stats!
#     weights = model_results.params / model_results.params.abs().sum()


#     print("\n\nFor " + os.path.basename(file) + ":\n")
#     print(weights)
#     print(model_results.summary())

In [ ]:
# # Hyperparameters for weight optimization.
# mean = 0
# std_dev = 1

# # The interval range for integration.
# inv_a = 0
# inv_b = 14

# # Returns the result of evaluating the Gaussian weight function at a given value `x`.
# # Dependant on the value of the `mean` and `std_dev` hyperparameters.
# def gauss_weight(x):
#     return norm.pdf(x, loc = mean, scale = std_dev)

# # The lagged cross-correlation is given by c_fg = ∫ f(x) * g(x + r) dx.
# # Thus, `f` and `g` are the respective functions to integrate, and `r` is the offset lag.
# # Returns c_fg(r), the result of the lagged cross-correlation of f, g, and r. 
# def lagged_cross_correlation(r, f, g):
#     # Calculate the lagged cross-correlation and return the result, integrating from x = a to x = b - r.
#     # Thus, assumes that the caller WLCC function adheres to an integration interval of [a, b].
#     integral_result, error = quad(func = (lambda x: f(x) * g(x + r)), a = 0, b = inv_b - r)
#     return integral_result

# # The integrand in use during the WLCC integrals.
# def integrand(r, f, g):
#     c_fg = lagged_cross_correlation(r, f, g)
#     w = gauss_weight(r)
#     return c_fg * w

# # Computes S_fg, the weighted lagged cross-correlation (WLCC) between two datasets.
# def weighted_lagged_cross_correlation(df_f, df_g):
#     # Interpolate both given dataframes into continuous functions on a given interval.
#     # Apparently, a cubic spline interpolator is best for a series of Gaussian-like curves.
#     func_f = interp1d(df_f.iloc[:, 0], df_f.iloc[:, 1], kind = 'cubic', bounds_error = False, fill_value = "extrapolate")
#     func_g = interp1d(df_f.iloc[:, 0], df_f.iloc[:, 1], kind = 'cubic', bounds_error = False, fill_value = "extrapolate")
#     # The numerator of the function is ∫ c_fg(r) * w(r) dr.
#     numerator, num_error = quad(func = (lambda r: integrand(r, func_f, func_g)), a = inv_a, b = inv_b)
#     # The denominator of the function is ((∫ c_ff(r) * w(r) dr) * (∫ c_gg(r) * w(r) dr)) ^ 1/2
#     denominator_f, den_f_error = quad(func = (lambda r: integrand(r, func_f, func_f)), a = inv_a, b = inv_b)
#     denominator_g, den_g_error = quad(func = (lambda r: integrand(r, func_g, func_g)), a = inv_a, b = inv_b)
#     denominator = (denominator_f * denominator_g) ** (1/2)
#     # Calculate S_fg from the numerator and denominator.
#     S_fg = numerator / denominator

#     return S_fg

In [ ]:
# # The `simulated_files` list is the list containing graphs of the simulated PXRD patterns.
# simulated_files = [f for f in os.listdir(fullgraph) if f.endswith("sim.chi")]

# # Split off the simulated PXRD patterns to keep the list solely of experimental data.
# experimental_files = [f for f in os.listdir(fullgraph) if f.endswith("exp.chi")]

# # Map the simulated file paths to their data in a dictionary.
# sim_data = {}
# for file_name in simulated_files:
#     path = os.path.join(fullgraph, file_name)
#     x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#     cif_data = np.column_stack((x_data, y_data))
#     cif_dataframe = pd.DataFrame(cif_data)
#     sim_data[file_name] = (x_data, y_data, cif_dataframe)

# # Prepare a container for the results of the correlation calculations.
# results = []

# # Loop over each file in the experimental files list, and for each file:
# for file in experimental_files:
#     # Load in the experimental file, unpack the data, and split it into an array.
#     path = os.path.join(fullgraph, file)
#     x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#     exp_data = np.column_stack((x_data, y_data))
#     exp_dataframe = pd.DataFrame(exp_data)
    
#     # Compute the experimental data's correlation to each simulated data.
#     row = {"exp_file": file}
#     for sim_fname, (sim_x, sim_y, sim_frame) in sim_data.items():
#         # Interpolate the simulated data onto the experimental grid.
#         y_interp = np.interp(x_data, sim_x, sim_y)
#         new_sim_frame = pd.DataFrame(np.column_stack((x_data, y_interp)))
#         # Calculate the WLCC from the experimental data and interpolated simulated data.
#         correlation = weighted_lagged_cross_correlation(exp_dataframe, new_sim_frame)
#         row[sim_fname] = correlation
#     results.append(row)

# # Print the mixture data for fact-checking.
# print("CIF files in mixture, with respective weights:")
# cif_files = mixture_results.get("chosen_cifs")
# cif_weights = mixture_results.get("weights")
# for i in range(len(cif_files)):
#     print("- " + os.path.basename(cif_files[i]) + ": " + str(cif_weights[i]))

# # Build a dataframe and save.
# df = pd.DataFrame(results).set_index("exp_file")
# df = df.div(df.sum(axis = 1), axis = 0)
# print(df.to_string(float_format="%.4f"))

# out_csv = os.path.join(fulldata, "correlation_weighted.csv")
# df.to_csv(out_csv)
# print(f"\nSaved correlations to {out_csv}.")

In [ ]:
# def single_pearson(ciffile: str):
#     simulated_files = [f for f in os.listdir(fullgraph) if f.endswith("sim.chi")]
#     simulated_files.sort()
#     experimental_files = [ciffile]
#     # Map the simulated file paths to their data in a dictionary.
#     sim_data = {}
#     for file_name in simulated_files:
#         path = os.path.join(fullgraph, file_name)
#         x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#         sim_data[file_name] = (x_data, y_data)
#     # Prepare a container for the results of the correlation calculations.
#     results = []
#     # Loop over each file in the experimental files list, and for each file:
#     for file in experimental_files:
#         # Load in the experimental file, unpack the data, and split it into an array.
#         path = os.path.join(fullgraph, file)
#         x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#         # Compute the experimental data's correlation to each simulated data.
#         row = {"exp_file": file}
#         for sim_fname, (sim_x, sim_y) in sim_data.items():
#             # Interpolate the simulated data onto the experimental grid.
#             y_interp = np.interp(x_data, sim_x, sim_y)
#             # Calculate the Pearson correlation from the experimental data and interpolated simulated data.
#             correlation = np.corrcoef(y_data, y_interp)[0,1]
#             row[sim_fname] = correlation
#         results.append(row)
#     # Build a dataframe and save.
#     df = pd.DataFrame(results).set_index("exp_file")
#     df = df.div(df.sum(axis = 1), axis = 0)
#     df = df.clip(0, 1)
#     return df

# results = single_pearson("CeO2-LaB6_mix00_exp.chi")
# print(results)

In [ ]:
# # Loop for testing both MLR and PC on larger amounts of data.
# def generate_samples():
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         cfg = SimConfig(
#                 wavelength = 0.1812,
#                 two_theta_range = (0.5, 15.0),
#                 n_points = 4096,
#             )
#         results = generate_mixture_batch(cif_paths = [ceo2_cif, lab6_cif], cfg = cfg, n_samples = 50)
#         ceo2_weights_actual = []
#         lab6_weights_actual = []
#         for i in range(len(results)):
#             ceo2_weights_actual.append(results[i].get("weights")[0])
#             lab6_weights_actual.append(results[i].get("weights")[1])
#             exp_data = np.column_stack((results[i].get("x"), results[i].get("y")))
#             if(i < 10):
#                 np.savetxt(os.path.join(fullgraph, "CeO2-LaB6_mix0" + str(i) + "_exp.chi"), exp_data, delimiter = ' ')
#             else:
#                 np.savetxt(os.path.join(fullgraph, "CeO2-LaB6_mix" + str(i) + "_exp.chi"), exp_data, delimiter = ' ')
#         actual_weights = np.column_stack((ceo2_weights_actual, lab6_weights_actual))
#         return actual_weights

# def pearson_correlation():
#     prev_time = time.time()
#     # The `simulated_files` list is the list containing graphs of the simulated PXRD patterns.
#     simulated_files = [f for f in os.listdir(fullgraph) if f.endswith("sim.chi")]
#     simulated_files.sort()
#     # Split off the simulated PXRD patterns to keep the list solely of experimental data.
#     experimental_files = [f for f in os.listdir(fullgraph) if f.endswith("exp.chi")]
#     experimental_files.sort()
#     # Making sure no odd file things happen
#     # Map the simulated file paths to their data in a dictionary.
#     sim_data = {}
#     for file_name in simulated_files:
#         path = os.path.join(fullgraph, file_name)
#         x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#         sim_data[file_name] = (x_data, y_data)
#     # Prepare a container for the results of the correlation calculations.
#     results = []
#     # Loop over each file in the experimental files list, and for each file:
#     for file in experimental_files:
#         # Load in the experimental file, unpack the data, and split it into an array.
#         path = os.path.join(fullgraph, file)
#         x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#         # Compute the experimental data's correlation to each simulated data.
#         row = {"exp_file": file}
#         for sim_fname, (sim_x, sim_y) in sim_data.items():
#             # Interpolate the simulated data onto the experimental grid.
#             y_interp = np.interp(x_data, sim_x, sim_y)
#             # Calculate the Pearson correlation from the experimental data and interpolated simulated data.
#             correlation = np.corrcoef(y_data, y_interp)[0,1]
#             row[sim_fname] = correlation
#         results.append(row)
#     # Build a dataframe and save.
#     df = pd.DataFrame(results).set_index("exp_file")
#     df = df.div(df.sum(axis = 1), axis = 0)
#     df = df.clip(0, 1)
#     print("Pearson calculation time: " + str(time.time() - prev_time))
#     return df

# def multiple_linear_regression():
#     prev_time = time.time()
#     # The `simulated_files` list is the list containing graphs of the simulated PXRD patterns.
#     simulated_files = [f for f in os.listdir(fullgraph) if f.endswith("sim.chi")]
#     simulated_files.sort()
#     # Split off the simulated PXRD patterns to keep the list solely of experimental data.
#     experimental_files = [f for f in os.listdir(fullgraph) if f.endswith("exp.chi")]
#     experimental_files.sort()
#     # Map the simulated file paths to their data in a dictionary.
#     sim_data = {}
#     for file_name in simulated_files:
#         path = os.path.join(fullgraph, file_name)
#         x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#         sim_data[file_name] = (x_data, y_data)
#     # Prepare a container for the results of the correlation calculations.
#     weights_result = 0
#     # Loop over each file in the experimental files list, and for each file:
#     for file in experimental_files:
#         aggregate_sim_data = 0
#         for sim_fname, (sim_x, sim_y) in sim_data.items():
#             # Load in the experimental file, unpack the data, and split it into an array.
#             path = os.path.join(fullgraph, file)
#             x_data, y_data = np.loadtxt(path, delimiter = " ", unpack = True)
#             # Interpolate the simulated data onto the experimental grid.
#             y_interp = np.interp(x_data, sim_x, sim_y)
#             # Add the simulated data onto the aggregate dataframe.
#             aggregate_sim_data = np.column_stack([aggregate_sim_data, y_interp]) if type(aggregate_sim_data) != int else y_interp
#         # Set up and fit the stats to an Ordinary Least Squares Model (for statistical MLR).
#         aggregate_sim_data = sm.add_constant(aggregate_sim_data)
#         model_results = sm.OLS(y_data, aggregate_sim_data).fit()
#         weights_normalized = model_results.params / np.sum(np.abs(model_results.params))
#         weights_result = np.row_stack([weights_result, [weights_normalized[1], weights_normalized[2]]]) if type(weights_result) != int else [weights_normalized[1], weights_normalized[2]]
#     # Return the correlations.
#     print("MLR calculation time: " + str(time.time() - prev_time))
#     return weights_result

# def run_batch():
#     actual_weights = generate_samples()
#     # pearson_results = pearson_correlation()
#     # pearson_results = pearson_results.to_numpy()
#     # results = np.column_stack([actual_weights, pearson_results])
#     mlr_weights_norm = multiple_linear_regression()
#     results = np.column_stack([actual_weights, mlr_weights_norm])
#     return results

# total = 0
# for i in range(10):
#     print("\nTrial " + str(i + 1) + " timing:")
#     result = run_batch()
#     total = result if type(total) == int else np.row_stack([total, result])

# df = pd.DataFrame(total, columns = ["CeO2 Weight", "LaB6 Weight", "MLR: CeO2", "MLR: LaB6"])
# df.to_csv("correlation_test_data", index = False)

In [ ]:
# # Creates and saves a simulated mixture of the sequence of CIF files `files` passed into it.
# # Returns a dictionary containing lists of the CIF files, their weights in the mixtures, and other information.
# def save_simulated_mix(files: list[str], index: int) -> Dict:
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         # Generate a simulated mixture pattern from the given CIF files.
#         simulated_mixture = generate_mixture_pattern(cif_paths = files, cfg = cfg)
#         # Consolidate the pattern results to extract the proper data.
#         exp_data = np.column_stack(tup = (simulated_mixture.get("x"), simulated_mixture.get("y")))
#         # Save the pattern's data to the graphs directory.
#         np.savetxt(fname = os.path.join(fullgraph, "mix" + str(index) + "_exp.chi"), X = exp_data, delimiter = ' ')
#         # Return the pattern's data for further use.
#         return simulated_mixture

In [ ]:
# aggregate_results = 0
# for i in range(10):
#     print("Beginning trial " + str(i + 1) + "...")
#     start_time = time.time()
#     for j in range(25):
#         results = approximate_pattern()
#         aggregate_results = results if type(aggregate_results) == int else pd.concat([aggregate_results, results], ignore_index = True)
#     end_time = time.time()
#     print("Trial had calculation time of " + str(end_time - start_time) + ".")
# aggregate_exp = pd.DataFrame(aggregate_experimental_data, columns = ["CeO2: Actual Lat.", "LaB6: Actual Lat.", "CeO2: Actual Weight", "LaB6: Actual Weight"])
# total_data = pd.concat(objs = [aggregate_exp, aggregate_results], axis = 1)
# total_data.to_csv("mlr_corr-lat_data_2.csv", index = False)

In [ ]:
# # Generate a new mixture pattern and approximate the lattice sizes and weights.
# # Also, prints out the resulting labeled dataframe.
# def approximate_pattern() -> pd.DataFrame:
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         # Generate the target pattern from the loaded-in CIF files.
#         target_pattern = generate_mixture_pattern(cif_paths = cif_files, cfg = cfg)
#         # Consolidate the pattern results to extract the proper data.
#         target_data = np.column_stack(tup = (target_pattern.get("x"), target_pattern.get("y")))
#         # Create a container for the lattice and correlation data for all candidates.
#         aggregate_correlation_data = []
#         # Find the lattice and correlation data for all of the candidate simulation files.
#         for i in range(len(cif_files)):
#             correlation_data = find_latlength_corr(target_pattern = target_data, base_pattern = cif_files[i])
#             aggregate_correlation_data.append(correlation_data)
#         # Find all of the peaks for possible correlation combinations by selecting only local maxima.
#         correlation_maxima = []
#         for i in range(len(aggregate_correlation_data)):
#             # For each data row in the aggregated correlation data:
#             current_data = aggregate_correlation_data[i]
#             candidate_maxima = []
#             for j in range(len(current_data)):
#                 # If the entry at the beginning of the list is greater than the one after it, or
#                 if(j == 0):
#                     if(current_data[j][1] > current_data[j + 1][1]):
#                         candidate_maxima.append(current_data[j])
#                 # If the entry at the end of the list is greater than the one before it, or
#                 elif(j == (len(current_data) - 1)):
#                     if(current_data[j - 1][1] < current_data[j][1]):
#                         candidate_maxima.append(current_data[j])
#                 # If the entry in any other position is greater than the entries around it, then add it to the maxima list.
#                 elif(current_data[j - 1][1] < current_data[j][1] and current_data[j][1] > current_data[j + 1][1]):
#                     candidate_maxima.append(current_data[j])
#             # Add the list of the candidate's correlation maxima to the aggregate list.
#             correlation_maxima.append(candidate_maxima)
#         # Compile the stats of the lattice and correlation data from all candidates.
#         lattice_sizes = []
#         total_weights = []
#         for i in range(len(correlation_maxima)):
#             # For each candidate file, process all of the lattice-correlation data.
#             lattice_size = 0
#             total_weight = 0
#             for j in range(len(correlation_maxima[i])):
#                 # Multiply each lattice size by their respective weight.
#                 lattice_size = lattice_size + correlation_maxima[i][j][0] * correlation_maxima[i][j][1]
#                 # Add the weight to the accumulator.
#                 total_weight = total_weight + correlation_maxima[i][j][1]
#             # If the weight of this candidate is statistically significant, then round off the lattice size and normalize the weight.
#             if(total_weight > 0):
#                 lattice_size = round(lattice_size / total_weight, 3)
#                 total_weight = total_weight / len(correlation_maxima[i])
#             # If it is not statistically significant, then set both the weight and lattice to zero.
#             else:
#                 lattice_size = 0
#                 total_weight = 0
#             # Add both the new lattice size and total weight to their resepective containers.
#             lattice_sizes.append(lattice_size)
#             total_weights.append(total_weight)
#         # Normalize the weights by dividing each by the total sum of the weights.
#         normalized_weights = []
#         for weight in total_weights:
#             normalized_weights.append(weight / sum(total_weights))
#         # Append the data together and construct a dataframe from the results.
#         results_array = np.append(lattice_sizes, normalized_weights)
#         results_frame = pd.DataFrame([[x] for x in results_array]).T
#         results_frame.columns = ["CeO2: Est. Lat.", "LaB6: Est. Lat.", "CeO2: Est. Weight", "LaB6: Est. Weight"]
#         # Return the resulting dataframe.
#         return results_frame

In [ ]:
# def latlength_corr_postprocessing(correlation_data: np.ndarray) -> Tuple[float, float]:
#     # Find all of the peaks for possible correlation combinations by selecting only local maxima.
#     candidate_maximum = 0
#     for i in range(len(correlation_data)):
#         # If the entry at the beginning of the list is greater than the one after it, or
#         if(i == 0):
#             if(correlation_data[i][1] > correlation_data[i + 1][1]):
#                 candidate_maxima.append(correlation_data[i])
#         # If the entry at the end of the list is greater than the one before it, or
#         elif(i == (len(correlation_data) - 1)):
#             if(correlation_data[i - 1][1] < correlation_data[i][1]):
#                 candidate_maxima.append(correlation_data[i])
#         # If the entry in any other position is greater than the entries around it, then add it to the maxima list.
#         elif(correlation_data[i - 1][1] < correlation_data[i][1] and correlation_data[i][1] > correlation_data[i + 1][1]):
#             candidate_maxima.append(correlation_data[i])
#     # Compile the stats of the lattice and correlation data from all candidates.
#     lattice_size = 0
#     total_weight = 0
#     # print("Potential peaks:")
#     for i in range(len(candidate_maxima)):
#         # print(candidate_maxima[i])
#         # Multiply each lattice size by their respective weight.
#         lattice_size = lattice_size + candidate_maxima[i][0] * candidate_maxima[i][1]
#         # Add the weight to the accumulator.
#         total_weight = total_weight + candidate_maxima[i][1]
#     # If the weight of this candidate is statistically significant, then round off the lattice size and normalize the weight.
#     if(total_weight > 0):
#         lattice_size = lattice_size / total_weight
#         total_weight = total_weight / len(candidate_maxima)
#     # If it is not statistically significant, then set both the weight and lattice to zero.
#     else:
#         lattice_size = 0
#         total_weight = 0
#     # Return the weight and lattice size.
#     return lattice_size, total_weight

In [ ]:
# # Calculates the correlation of CIF file `base_pattern` with the mixture `target_pattern` via Multiple Linear Regression over a series of lattice lengths.
# # Returns a list of tuples of the form ([lattice-size], [correlation-to-target]).
# def find_latlength_corr(target_pattern: np.ndarray, base_pattern: str) -> List[Tuple[float, float]]:
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         # Obtain the lattice from the given candidate CIF file.
#         lattice = get_lattice(base_pattern)
#         # Obtain the upper and lower bounds of the lattice from its original size ([-0.5%, +1.0%]).
#         lower_bound = lattice.abc[0] - lattice.abc[0] * 0.005
#         upper_bound = lattice.abc[0] + lattice.abc[0] * 0.02
#         # Normalize the experimental data to be in line with the simulated data (scaling with the greatest dataset's value).
#         base_sim = save_simulated_pattern(cif_file = base_pattern, lattice_sizes = (lattice.abc[0], lattice.abc[0], lattice.abc[0]), append = "base")
#         max_sim_peak = np.max(base_sim[:, 1])
#         max_exp_peak = np.max(target_pattern[:, 1])
#         scale_factor = max_sim_peak / max_exp_peak
#         target_pattern[:, 1] = target_pattern[:, 1] * scale_factor
#         # Extract the baseline from this mystery pattern.
#         baseline = extract_baseline(mystery_pattern = target_pattern)
#         # Calculate the correlations between the target CIF and the simulated CIFs of various lattice sizes
#         lattices_and_correlations = []
#         # Iterate over the possible values of the lattice size with a set step size.
#         i = lower_bound
#         step_size = 0.001
#         while(i <= upper_bound):
#             # Round-off to avoid floating-point errors.
#             i = round(i, 3)
#             # Create a simulated pattern with specific lattice size.
#             sim_pattern = save_simulated_pattern(cif_file = base_pattern, lattice_sizes = (i, i, i), append = str(i))
#             # Interpolate the data of the simulated pattern to that of the target pattern.
#             pchip_interpolator = PchipInterpolator(sim_pattern[:, 0], sim_pattern[:, 1])
#             y_interpolated = pchip_interpolator(target_pattern[:, 0])
#             y_interpolated = y_interpolated + baseline
#             # Add the interpolated data to the OLS model.
#             sim_data = sm.add_constant(y_interpolated)
#             # Fit the OLS model and extract the correlation data from the results.
#             model_results = sm.OLS(target_pattern[:, 1], sim_data).fit()
#             correlation = model_results.params
#             # Reset the OLS model from further use.
#             model_results.remove_data()
#             # Append the lattice size and corresponding correlation to the aggregate data.
#             lattices_and_correlations.append((i, correlation[1]))
#             i = i + step_size
#         # Return the aggregate collection of all lattice sizes and correlations.
#         return lattices_and_correlations

In [ ]:
# # It's extrapolation time! Let's get some temperature data!
# # Obtain the test set from the specified directory.
# test_files = glob.glob(constantpath + "*.xy")
# # Create a dataframe to hold the results of the estimation.
# results_frame = pd.DataFrame(columns = ["File Name", "Estimated Lattice Size", "Extrapolated Temperature"])
# # Iterate through the test files and estimate the temperature for each.
# for file in test_files:
#     print("Examining file " + str(os.path.basename(file)) + "...")
#     # The test row should have the file's name,
#     test_row = []
#     test_row.append(os.path.basename(file))
#     lat_size = approximate_lattice(file)
#     # The estimated lattice size, and
#     test_row.append(lat_size)
#     # The extrapolated temperature.
#     est_temperature = extrapolator(lat_size)
#     test_row.append(est_temperature)
#     # Append this row to the end of the dataframe.
#     results_frame.loc[len(results_frame)] = test_row
# print("\n\nEstimation Results:")
# print(results_frame)

# results_frame.to_csv("lat-and-temp-estimation_data_nelder-mead-niter-2.csv")

In [ ]:
# def find_weight(exp_data: np.ndarray, base_pattern: str = base_ceo2_pattern, lattice_param: float = 5.410) -> float:
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         # Get datasets for both the CeO2 pattern and the unknown mixture.
#         sim_data = save_simulated_pattern(cif_file = base_pattern, lattice_sizes = (lattice_param, lattice_param, lattice_param))

#         max_sim_peak = np.max(sim_data[:, 1])
#         max_exp_peak = np.max(exp_data[:, 1])
#         scale_factor = max_sim_peak / max_exp_peak
#         exp_data[:, 1] = exp_data[:, 1] * scale_factor

        
#         # Interpolate the CeO2 pattern onto the unknown mixture's x-axis.
#         pchip_interpolator = PchipInterpolator(sim_data[:, 0], sim_data[:, 1])
#         y_interpolated = pchip_interpolator(exp_data[:, 0])
#         # Extract the peak locations and intensities for the CeO2 pattern.
#         sim_peaks_locations = find_peaks(y_interpolated)[0]
#         sim_peaks_intensities = []
#         for peak_index in sim_peaks_locations:
#             sim_peaks_intensities.append(y_interpolated[peak_index])
#         # Extract the intensities of the unknown mixture at the same locations.
#         exp_peaks = []
#         for peak_index in sim_peaks_locations:
#             exp_peaks.append(exp_data[:, 1][peak_index])
#         # Compare the intensities and find the {mean, mode} of the scaling factor to figure out the weight of the CeO2 in the mixture.
#         peak_scalings =  np.array(sim_peaks_intensities) / np.array(exp_peaks)
#         print("PEAK SCALINGS:")
#         print(peak_scalings)
#         print(" - mean = " + str(np.mean(peak_scalings)))
#         return np.mean(peak_scalings)

In [ ]:
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     test_files = glob.glob(constantpath + "*.xy")[1]
#     test_data = save_experimental_pattern(test_files)
#     lattice = approximate_lattice(test_files)
#     # mixture_properties = generate_mixture_pattern(cif_paths = [base_ceo2_pattern, base_lab6_pattern], cfg = cfg)

#     # weight_result = find_weight(exp_data = np.column_stack(tup = [mixture_properties.get("x"), mixture_properties.get("y")]))
#     weight_result = find_weight(exp_data = test_data, lattice_param = lattice)

#     print("Simulated testing results:")
#     # print(mixture_properties.get("chosen_cifs"))
#     # print(mixture_properties.get("weights"))
#     print("CeO2 weight = " + str(weight_result))

In [ ]:
# As the XY files remained the same throughout repeated run-throughs of the code, this has been elided to speed up runtime.
# However, if `dat_files` changes at all, this should be put back in for proper behavior.

# # Create the simulated patterns of the XY files, and save the data for later. 
# for file in dat_files:
#     save_experimental_pattern(file)

In [ ]:
# As the CIF files remained the same throughout repeated run-throughs of the code, this has been elided to speed up runtime.
# However, if `cif_files` changes at all, this should be put back in for proper behavior.

# # Create the simulated patterns of the CIF files, and save the data for later. 
# for file in cif_files:
#     save_simulated_pattern(file)

In [ ]:
# # Obtain the simulated CIF files from the proper directory, and extract them to a list for later use.
# # Currently, this should only have a CeO2 CIF and a LaB6 CIF.
# print("Simulated data files:")
# cif_files = glob.glob(cifpath + "*.cif")
# for file in cif_files:
#     print("Loaded in \"" + os.path.basename(file) + "\".")

# # Obtain the experimental XY files from the proper directory, and extract them to a list for later use.
# print("Experimental data files:")
# dat_files = glob.glob(datapath + "*.xy")
# for file in dat_files:
#     print("Loaded in \"" + os.path.basename(file) + "\".")

In [ ]:
# # Create an extrapolator for determining temperature as a function of estimated lattice size.
# # Constant data taken from the "Pt_thermal_expansion" spreadsheet.
# ceo2_lat_percent_constants = [-0.17, -0.085, 0, 0.106, 0.21, 0.321, 0.437, 0.558, 0.683, 0.813, 0.946, 1.083, 1.223, 1.365, 1.51, 1.657]
# ceo2_temp_constants = [100, 200, 293, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600]
# # Translate the lattice percentage data into more usable lattice parameter data.
# ceo2_lattice_constants = []
# for percentage in ceo2_lat_percent_constants:
#     ceo2_lattice_constants.append((1 + percentage / 100) * 5.411651)
# # Create an interpolator function from the given lattice parameters and corresponding temperature values.
# extrapolator = interp1d(ceo2_lattice_constants, ceo2_temp_constants, kind = 'cubic', fill_value = 'extrapolate')

In [ ]:
# # The driver for the above cells. Creates a dataset of three-phase mixtures and calculates the shifted and unshifted correlation for them.
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     timing_data = []
#     # Extract the CIF files from the proper directory and put them into a list.
#     ceo2_cif = glob.glob(cifpath + "*CeO2*.cif")[0]
#     lab6_cif = glob.glob(cifpath + "*LaB6*.cif")[0]
#     ni_cif = glob.glob(cifpath + "*Ni*.cif")[0]
#     cif_list = [ceo2_cif, lab6_cif, ni_cif]
#     prev_time = time.time()
#     # Generate a bunch of randomly-weighted mixtures from the given CIF list.
#     mixture_data = generate_random_mixtures(cif_list = cif_list, n = 50)
#     aggregate_mixture_data = []
#     for mixture in mixture_data:
#         mixture_data_row = []
#         mixture_data_row.append(mixture.get("filename"))
#         mixture_data_row = np.concatenate((mixture_data_row, mixture.get("lattices"), mixture.get("weights")))
#         aggregate_mixture_data.append(mixture_data_row)
#     aggregate_mixture_data = np.array(aggregate_mixture_data)
#     timing_data.append(time.time() - prev_time)
#     print("Mixtures generated. Calculating correlations...")
#     prev_time = time.time()
#     # Find the correlation data between the mixtures and the component CIFs (unshifted).
#     unshifted_results =  calculate_correlation_unshifted(mixture_list = mixture_data, cif_list = cif_list)
#     timing_data.append(time.time() - prev_time)
#     print("Unshifted correlations calculated. Calculating shifted correlations...")
#     prev_time = time.time()
#     # Find the correlation data between the mixtures and the component CIFs (shifted).
#     shifted_results = calculate_highest_correlation(mixture_list = mixture_data, cif_list = cif_list)
#     timing_data.append(time.time() - prev_time)
#     print("Shifted calculations calculated. Performing final data processing...")
#     prev_time = time.time()
#     # Stick the unshifted and shifted correlation data together.
#     final_results = np.column_stack(tup = [aggregate_mixture_data, unshifted_results, shifted_results])
#     # Save the final results to a CSV file.
#     np.savetxt('mixture_lat-corr_data_extra2.csv', final_results, delimiter = ',', fmt = "%s")
#     timing_data.append(time.time() - prev_time)
#     print(final_results)
#     print("Timing data for: [Mixture generation, unshifted correlation, shifted correlation, final processing]:")
#     print(timing_data)

In [ ]:
# # Make a dataset of three-phase mixtures, with each phase having a randomly-determined lattice size and weight.
# # Returns a series of dictionaries (one for each mixture) containing data points, component weights, and lattice sizes.
# def generate_random_mixtures(cif_list: list[str], n: int = 50) -> list[dict]:
#     mixture_data = generate_mixture_batch(cif_paths = cif_list, cfg = cfg, n_samples = n, n_phases_range = (3, 3))
#     return mixture_data

In [ ]:
# # Calculate the correlation between the base component CIFs in `cif_list` and the three-phase mixtures in `mixture_list`.
# def calculate_correlation_unshifted(mixture_list: list[dict], cif_list: list[str]) -> np.ndarray:
#     # Create a container for all of the correlation data.
#     full_correlations = []
#     # For each simulated mixture in the given mixture list:
#     for mixture in mixture_list:
#         # Extract the relevant data from the mixture's dictionary.
#         mixture_data = np.column_stack(tup = [mixture.get("x"), mixture.get("y")])
#         # Create a container for holding the predictions for each component.
#         mixture_correlations = []
#         # For each CIF file given in the file list:
#         for component_cif in cif_list:
#             # Obtain the component data from the CIF while approximating the best lattice size.
#             component_data = save_simulated_pattern(component_cif)
#             # Create an interpolator and use it to estimate the CIF's data on the mixture's x-axis.
#             pchip_interpolator = PchipInterpolator(component_data[:, 0], component_data[:, 1])
#             y_interpolated = pchip_interpolator(mixture_data[:, 0])
#             # Add the interpolated data to the OLS model.
#             sim_model = sm.add_constant(y_interpolated)
#             # Fit the OLS model and extract the correlation data from the results.
#             model_results = sm.OLS(mixture_data[:, 1], sim_model).fit()
#             correlation = model_results.params
#             # Append the resulting data to the respective containers.
#             mixture_correlations.append(correlation[1])
#             # Reset the OLS model from further use.
#             model_results.remove_data()
#         # Normalize the correlation data to more accurately reflect actual weights.
#         normalized_correlations = []
#         clamped_data = np.clip(mixture_correlations, a_min = 0, a_max = 1).tolist()
#         for correlation in clamped_data:
#             normalized_correlations.append(correlation / np.sum(clamped_data))
#         # Add the normalized correlations onto the full resulting dataset.
#         full_correlations.append(normalized_correlations)
#     return np.array(full_correlations)

In [ ]:
# # A list of lattice parameters for data collection.
# lattice_list = [(8.20678, 8.20678, 8.20678), (4.78879, 4.78879, 13.0814), (4.25362, 4.25362, 4.25362), (5.17946, 5.17946, 5.17946), (4.62, 4.62, 4.62)]

In [ ]:
# # The driver for the above cells. Creates a dataset of three-phase mixtures and calculates the shifted and unshifted correlation for them.
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     # Extract the CIF files from the proper directory and put them into a list.
#     cif_list = glob.glob(cifpath + "*.cif")
#     trimmed_cif_list = []
#     # Create the labels for an axis of the resulting dataframe.
#     trimmed_cif_list = []
#     for cif in cif_list:
#         trimmed_cif_list.append(os.path.basename(cif))
#     column_labels = np.concatenate((trimmed_cif_list, trimmed_cif_list))
#     # Create a container for the aggregate results.
#     aggregate_mixture_data = []
#     # Extract the experimental files from the correct location.
#     exp_data_list = glob.glob(constantpath + "*.xy")[15:32]
#     trimmed_exp_list = []
#     # Create labels for another axis of the resulting dataframe.
#     for exp in exp_data_list:
#         trimmed_exp_list.append(os.path.basename(exp))
#     # For each file listed in the file list:
#     plt.figure()
#     plt.title("Spinel Mixture Data")
#     plt.xlabel("2θ")
#     plt.ylabel("Intensity")
#     for i in range(len(exp_data_list)):
#         # Extract the data from the experimental pattern (`skiprows` is used for skipping the header).
#         mixture_data = save_experimental_pattern(exp_data_list[i], skiprows = 6)
#         plt.plot(mixture_data[:, 0], mixture_data[:, 1] + (25 * i))
#         # Find the correlation data between the mixtures and the component CIFs (shifted).
#         shifted_results = calculate_highest_correlation(mixture_data = mixture_data, cif_list = cif_list, calculate_lattice = False)
#         # Add this correlation to the aggregate data container.
#         aggregate_mixture_data.append(shifted_results)
#     # Stick the unshifted and shifted correlation data together.
#     final_frame = pd.DataFrame(aggregate_mixture_data)
#     # Save the final results to a CSV file.
#     final_frame.columns = column_labels
#     final_frame.index = trimmed_exp_list
#     print("Final Results:")
#     print(final_frame)
#     final_frame.to_csv("spinel_lattice_more.csv")